# 第 16 节：重要性采样 (Importance Sampling)

---

## 📍 本节位置

```
策略梯度 (11-12) → Actor-Critic (14) → **重要性采样 (16)** → GAE (18) → PPO (19)
                                              ↑
                                          你在这里
```

重要性采样是 off-policy 强化学习的数学基石。它让我们能够用行为策略收集的数据来估计目标策略的期望，从而打破 on-policy 算法必须丢弃旧数据的限制。


## 为什么 Off-Policy 学习需要修正？

### On-Policy vs Off-Policy

| 特性 | On-Policy | Off-Policy |
|------|-----------|------------|
| 数据来源 | 当前策略 $\pi$ | 行为策略 $\mu$（可能不同） |
| 样本效率 | 低（用完即弃） | 高（复用历史数据） |
| 稳定性 | 高 | 低（分布不匹配） |
| 典型算法 | REINFORCE, PPO | Q-Learning, DQN, SAC |

### 核心问题：分布不匹配

假设我们想计算 $J(\theta) = \mathbb{E}_{x \sim \pi_\theta}[f(x)]$，但我们只有 $x \sim \mu(\cdot)$ 的样本。

**直接使用 $\mu$ 的样本估计 $\pi$ 的期望是有偏的**，因为采样分布不同。

### 直觉理解

> 如果 $\mu$ 在某个区域采样不足，而 $\pi$ 认为该区域很重要，那么直接使用 $\mu$ 的样本会低估该区域的影响。

我们需要一种方法来**修正采样分布之间的差异**——这就是重要性采样的作用。


## 重要性采样的核心公式

### 数学推导

假设 $p(x)$ 和 $q(x)$ 是两个概率分布，且 $q(x) > 0$ 时 $p(x) > 0$（支撑集包含条件）。

我们想计算 $\mathbb{E}_{x \sim p}[f(x)]$，但只能从 $q$ 中采样：

$$
\begin{aligned}
\mathbb{E}_{x \sim p}[f(x)]
&= \int f(x) \, p(x) \, dx \\
&= \int f(x) \, \frac{p(x)}{q(x)} \, q(x) \, dx \\
&= \mathbb{E}_{x \sim q}\left[\frac{p(x)}{q(x)} \, f(x)\right]
\end{aligned}
$$

### 重要性权重 (Importance Weight)

$$\rho(x) = \frac{p(x)}{q(x)}$$

称为**重要性权重**。它校正了采样分布 $q$ 和目标分布 $p$ 之间的差异。

### 经验估计

给定 $N$ 个样本 $x_i \sim q(\cdot)$：

$$
\hat{\mathbb{E}}_{\text{IS}} = \frac{1}{N} \sum_{i=1}^{N} \rho(x_i) \, f(x_i)
$$

这是一个**无偏估计**（证明见下）：
$$\mathbb{E}_{q}[\rho(x) f(x)] = \mathbb{E}_{p}[f(x)]$$


## 轨迹级重要性采样 (Trajectory-level IS)

在强化学习中，我们需要估计目标策略 $\pi$ 下的期望回报，但只有行为策略 $\mu$ 采样的轨迹。

### 轨迹概率

一条轨迹 $\tau = (s_0, a_0, r_1, s_1, a_1, ..., s_{T-1}, a_{T-1}, r_T, s_T)$ 的概率：

**目标策略 $\pi$ 下**：
$$P(\tau \mid \pi) = \underbrace{d_0(s_0)}_{\text{初始状态}} \cdot \prod_{t=0}^{T-1} \underbrace{\pi(a_t \mid s_t)}_{\text{动作概率}} \cdot \underbrace{P(s_{t+1} \mid s_t, a_t)}_{\text{转移概率}}$$

**行为策略 $\mu$ 下**：
$$P(\tau \mid \mu) = d_0(s_0) \cdot \prod_{t=0}^{T-1} \mu(a_t \mid s_t) \cdot P(s_{t+1} \mid s_t, a_t)$$

### 轨迹重要性权重

$$
\rho_{0:T-1}(\tau) = \frac{P(\tau \mid \pi)}{P(\tau \mid \mu)} = \prod_{t=0}^{T-1} \frac{\pi(a_t \mid s_t)}{\mu(a_t \mid s_t)}
$$

注意到**环境动态 $P$ 和 $d_0$ 被约掉了**！这意味着我们不需要知道环境模型，只需要知道两个策略的动作概率之比。

### 应用

$$\mathbb{E}_{\tau \sim \pi}[G_0] = \mathbb{E}_{\tau \sim \mu}\left[\left(\prod_{t=0}^{T-1} \frac{\pi(a_t \mid s_t)}{\mu(a_t \mid s_t)}\right) \cdot G_0\right]$$

这个公式是 **Off-Policy Policy Gradient** 的核心理论基础。


## 每决策重要性采样 (Per-decision IS)

### 问题：轨迹级 IS 的方差过大

轨迹级 IS 的权重是多个概率比值的乘积，随着轨迹长度增长，权重方差会指数级增长：

$$\text{Var}\left[\prod_{t=0}^{T-1} \frac{\pi(a_t \mid s_t)}{\mu(a_t \mid s_t)}\right] \propto \text{指数增长}$$

### Per-decision IS：更细粒度的校正

对于回报 $G_t = \sum_{k=t}^{T-1} \gamma^{k-t} r_{k+1}$，**奖励 $r_{k+1}$ 只依赖于 $s_k, a_k, s_{k+1}$**。

因此我们只需要从 $t$ 到 $k$ 的重要性权重：

$$\mathbb{E}_{\pi}[r_{k+1}] = \mathbb{E}_{\mu}\left[\left(\prod_{j=t}^{k} \frac{\pi(a_j \mid s_j)}{\mu(a_j \mid s_j)}\right) \cdot r_{k+1}\right]$$

### 更精确的回报估计

$$\mathbb{E}_{\pi}[G_t] = \mathbb{E}_{\mu}\left[\sum_{k=t}^{T-1} \left(\prod_{j=t}^{k} \frac{\pi(a_j \mid s_j)}{\mu(a_j \mid s_j)}\right) \gamma^{k-t} r_{k+1}\right]$$

**优势**：每个奖励只乘以从 $t$ 到该奖励时刻的权重，而不是整条轨迹的权重，大幅降低方差。

**直观理解**：远处的奖励受更多策略决策影响，所以需要更多的校正；近处的奖励只需要很少的校正。

### 在 Off-Policy Actor-Critic 中的应用

许多现代 off-policy 算法（如 SAC、TD3）使用 Per-decision IS 的思想，通过截断或加权的方式来控制方差。


## 方差问题：为什么 IS 有高方差？

### 理论分析

重要性采样估计虽然无偏，但方差可能非常大：

$$\text{Var}_{q}[\rho(x)f(x)] = \mathbb{E}_{q}[\rho(x)^2 f(x)^2] - \underbrace{(\mathbb{E}_{p}[f(x)])^2}_{\text{常数}}$$

当 $ho(x) = p(x)/q(x)$ 很大时（即 $q$ 在 $p$ 有高密度的区域采样不足），方差会爆炸。

### 方差膨胀因子

$$\text{Var}_{q}[\rho(x) f(x)] = \text{Var}_{p}[f(x)] + \underbrace{\mathbb{E}_{p}\left[\left(\frac{p(x)}{q(x)} - 1\right) \text{Var}_{p}[f(x) \mid x]\right]}_{\text{额外方差项}}$$

当 $p = q$ 时，额外方差为 0。当 $p$ 和 $q$ 差异很大时，方差急剧增大。

### 诊断工具：有效样本量 (ESS)

$$\text{ESS} = \frac{\left(\sum_{i=1}^{N} \rho_i\right)^2}{\sum_{i=1}^{N} \rho_i^2} \approx \frac{N}{1 + \text{Var}_{q}[\rho]}$$

- ESS 衡量实际有效的独立样本数量
- 权重方差越大，ESS 越小
- 当 $\text{ESS} \ll N$ 时，IS 估计不可靠

### 在 RL 中的影响

在 RL 中，策略更新会导致 $\pi$ 和 $\mu$ 逐渐偏离，权重方差随之增大。这意味着：
1. 不能重用太旧的数据
2. 需要限制策略更新幅度（PPO 的核心动机）
3. 需要截断或平滑重要性权重


In [ ]:
# ============================================================
# 重要性采样演示：高斯分布
# ============================================================
# 目标：用 q 的样本估计 E_{x~p}[f(x)]
# 其中 p = N(1, 1), q = N(0, 2)，f(x) = x^2

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# 定义分布参数
mu_p, sigma_p = 1.0, 1.0   # 目标分布 p ~ N(1, 1)
mu_q, sigma_q = 0.0, 2.0   # 行为分布 q ~ N(0, 2)

def f(x):
    return x ** 2  # 我们想估计 E_{p}[x^2]

# 解析解
E_p_f = mu_p**2 + sigma_p**2  # E[x^2] = Var[x] + E[x]^2 = 1 + 1 = 2
print(f"真实值 E_{{x~p}}[x^2] = {E_p_f:.4f}")

# 1. 直接使用 q 的样本（有偏估计）
N = 10000
x_q = np.random.normal(mu_q, sigma_q, N)
direct_estimate = np.mean(f(x_q))
print(f"直接使用 q 的样本: {direct_estimate:.4f}  (偏差 = {direct_estimate - E_p_f:.4f})")

# 2. 重要性采样校正
# 权重: rho(x) = p(x) / q(x)
def gaussian_pdf(x, mu, sigma):
    return 1.0 / (sigma * np.sqrt(2 * np.pi)) * np.exp(-0.5 * ((x - mu) / sigma)**2)

rho = gaussian_pdf(x_q, mu_p, sigma_p) / gaussian_pdf(x_q, mu_q, sigma_q)
is_estimate = np.mean(rho * f(x_q))
print(f"重要性采样估计: {is_estimate:.4f}  (偏差 = {is_estimate - E_p_f:.4f})")

# 3. 蒙特卡洛直接估计（从 p 采样，作为基准）
x_p = np.random.normal(mu_p, sigma_p, N)
mc_estimate = np.mean(f(x_p))
print(f"MC 直接估计（从 p 采样）: {mc_estimate:.4f}  (偏差 = {mc_estimate - E_p_f:.4f})")

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 左图：分布和权重
x_grid = np.linspace(-8, 8, 500)
axes[0].plot(x_grid, gaussian_pdf(x_grid, mu_p, sigma_p), 'b-', label='p (目标)', linewidth=2)
axes[0].plot(x_grid, gaussian_pdf(x_grid, mu_q, sigma_q), 'r--', label='q (行为)', linewidth=2)
axes[0].set_xlabel('x'); axes[0].set_ylabel('密度')
axes[0].set_title('目标分布 p vs 行为分布 q')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# 中图：权重分布
axes[1].hist(rho, bins=50, alpha=0.7, edgecolor='black')
axes[1].axvline(1.0, color='r', linestyle='--', label='ρ=1 (完美匹配)')
axes[1].set_xlabel('重要性权重 ρ(x)'); axes[1].set_ylabel('频数')
axes[1].set_title(f'权重分布 (ESS/N = {(np.sum(rho)**2 / np.sum(rho**2)) / N:.3f})')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# 右图：估计值对比
methods = ['直接估计 (q)', 'IS 校正', 'MC (p)', '真实值']
values = [direct_estimate, is_estimate, mc_estimate, E_p_f]
colors = ['red', 'green', 'blue', 'black']
bar_pos = axes[2].bar(methods, values, color=colors, alpha=0.7)
axes[2].axhline(E_p_f, color='black', linestyle='--', linewidth=2)
axes[2].set_ylabel('E[x²]'); axes[2].set_title('估计值对比')
axes[2].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(values):
    axes[2].text(i, v + 0.05, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.savefig('outputs/figures/16_is_gaussian_demo.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/16_is_gaussian_demo.png")


In [ ]:
# ============================================================
# 重要性采样在 2 状态 MDP 中的应用
# ============================================================
# 环境：2 个状态 (s0, s1)，2 个动作 (a0, a1)
# 目标：演示 IS 权重的分布和方差

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# 定义简单 MDP
# 状态 0 → 动作 0 → 状态 0 (r=0)，动作 1 → 状态 1 (r=1)
# 状态 1 → 动作 0 → 状态 1 (r=0)，动作 1 → 状态 0 (r=1)
n_states = 2
n_actions = 2

# 行为策略 mu: 偏向于选择动作 0
mu = np.array([
    [0.9, 0.1],  # 状态 0: 90% 选 a0, 10% 选 a1
    [0.8, 0.2],  # 状态 1: 80% 选 a0, 20% 选 a1
])

# 目标策略 pi: 偏向于选择动作 1
pi = np.array([
    [0.2, 0.8],  # 状态 0: 20% 选 a0, 80% 选 a1
    [0.3, 0.7],  # 状态 1: 30% 选 a0, 70% 选 a1
])

# 转移矩阵 P[s, a, s']
P = np.zeros((n_states, n_actions, n_states))
P[0, 0, 0] = 1.0  # s0, a0 -> s0
P[0, 1, 1] = 1.0  # s0, a1 -> s1
P[1, 0, 1] = 1.0  # s1, a0 -> s1
P[1, 1, 0] = 1.0  # s1, a1 -> s0

# 奖励 R[s, a]
R = np.array([
    [0.0, 1.0],  # 状态 0
    [0.0, 1.0],  # 状态 1
])

def generate_trajectory(policy, max_steps=10):
    '''生成一条轨迹'''
    states, actions, rewards = [], [], []
    s = np.random.choice(n_states)  # 随机初始状态
    for _ in range(max_steps):
        a = np.random.choice(n_actions, p=policy[s])
        s_next = np.random.choice(n_states, p=P[s, a])
        r = R[s, a]
        states.append(s); actions.append(a); rewards.append(r)
        s = s_next
    return states, actions, rewards

# 生成多条轨迹
n_trajectories = 2000
max_steps = 10

all_weights = []
all_cumulative_returns = []

for _ in range(n_trajectories):
    states, actions, rewards = generate_trajectory(mu, max_steps)

    # 计算轨迹级重要性权重
    weight = 1.0
    for s, a in zip(states, actions):
        weight *= pi[s, a] / mu[s, a]

    # 计算累积回报
    G = sum(rewards)

    all_weights.append(weight)
    all_cumulative_returns.append(G)

all_weights = np.array(all_weights)
all_cumulative_returns = np.array(all_cumulative_returns)

# IS 估计
is_estimate_returns = np.mean(all_weights * all_cumulative_returns)

# 直接估计（使用 mu 的样本 — 有偏）
direct_estimate = np.mean(all_cumulative_returns)

print(f"=== 2 状态 MDP 结果 ===")
print(f"行为策略下直接估计: {direct_estimate:.4f}")
print(f"IS 校正后估计:      {is_estimate_returns:.4f}")
print(f"权重统计:")
print(f"  均值:  {np.mean(all_weights):.4f}")
print(f"  标准差: {np.std(all_weights):.4f}")
print(f"  最小值: {np.min(all_weights):.4f}")
print(f"  最大值: {np.max(all_weights):.4f}")
print(f"  ESS/N: {(np.sum(all_weights)**2 / np.sum(all_weights**2)) / n_trajectories:.4f}")

# 可视化权重分布
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(all_weights, bins=50, alpha=0.7, edgecolor='black', log=True)
axes[0].axvline(1.0, color='r', linestyle='--', label='ρ=1')
axes[0].set_xlabel('重要性权重 ρ')
axes[0].set_ylabel('频数 (log scale)')
axes[0].set_title(f'权重分布 (轨迹级, T={max_steps})')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# 权重 vs 回报
axes[1].scatter(all_weights, all_cumulative_returns, alpha=0.3, s=5)
axes[1].axvline(1.0, color='r', linestyle='--', alpha=0.5)
axes[1].set_xlabel('重要性权重 ρ'); axes[1].set_ylabel('累积回报 G')
axes[1].set_title('权重 vs 回报')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/figures/16_is_2state_mdp.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/16_is_2state_mdp.png")


## Weighted IS vs Ordinary IS

### Ordinary Importance Sampling (OIS)

$$\hat{J}_{\text{OIS}} = \frac{1}{N} \sum_{i=1}^{N} \rho_i \, f(x_i)$$

- **无偏**：$\mathbb{E}[\hat{J}_{\text{OIS}}] = J$
- **高方差**：特别是当权重分布偏斜时

### Weighted Importance Sampling (WIS)

$$\hat{J}_{\text{WIS}} = \frac{\sum_{i=1}^{N} \rho_i \, f(x_i)}{\sum_{i=1}^{N} \rho_i} = \sum_{i=1}^{N} \frac{\rho_i}{\sum_j \rho_j} \, f(x_i)$$

- **有偏**（但渐近无偏）：$\mathbb{E}[\hat{J}_{\text{WIS}}] \neq J$ 但 $\lim_{N \to \infty} \hat{J}_{\text{WIS}} \to J$
- **方差更低**：权重被归一化，极端值的影响被限制

### 对比

| 特性 | OIS | WIS |
|------|-----|-----|
| 偏差 | 无偏 | 有偏（$O(1/N)$）|
| 方差 | 高 | 低 |
| 是否在 $[x_{\min}, x_{\max}]$ 内 | ❌ | ✅（凸组合） |
| 渐近性质 | $\sqrt{N}$ 一致性 | $\sqrt{N}$ 一致性 |

### 在强化学习中的实际选择

**在策略梯度中，我们通常使用 OIS**（无偏性更重要）：
- Policy Gradient Theorem 依赖于梯度的无偏估计
- 引入偏差可能导致策略收敛到次优解

**在价值函数估计中，WIS 更常见**：
- In Q-Learning 中，WIS 能提供更稳定的目标值
- 牺牲少量偏差换取大幅方差降低


In [ ]:
# ============================================================
# OIS vs WIS：偏差和方差比较实验
# ============================================================
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# 设置：p 和 q 分布差异较大
# 目标：估计 E_{x~p}[f(x)], f(x) = x
# p = N(2, 0.8), q = N(0, 1.5)

mu_p, sigma_p = 2.0, 0.8
mu_q, sigma_q = 0.0, 1.5

def gaussian_pdf(x, mu, sigma):
    return 1.0 / (sigma * np.sqrt(2 * np.pi)) * np.exp(-0.5 * ((x - mu) / sigma)**2)

true_Ep = mu_p  # E[x] under p
f = lambda x: x

n_runs = 500
sample_sizes = [50, 100, 200, 500, 1000, 2000]

results_ois = {n: [] for n in sample_sizes}
results_wis = {n: [] for n in sample_sizes}

for n in sample_sizes:
    for run in range(n_runs):
        x_q = np.random.normal(mu_q, sigma_q, n)
        rho = gaussian_pdf(x_q, mu_p, sigma_p) / gaussian_pdf(x_q, mu_q, sigma_q)
        fx = f(x_q)

        # OIS
        ois_est = np.mean(rho * fx)
        results_ois[n].append(ois_est)

        # WIS
        wis_est = np.sum(rho * fx) / np.sum(rho)
        results_wis[n].append(wis_est)

# 计算偏差和方差
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ns = list(sample_sizes)
ois_bias = [np.mean(results_ois[n]) - true_Ep for n in ns]
wis_bias = [np.mean(results_wis[n]) - true_Ep for n in ns]
ois_var = [np.var(results_ois[n]) for n in ns]
wis_var = [np.var(results_wis[n]) for n in ns]

axes[0].plot(ns, ois_bias, 'b-o', label='OIS (无偏)', linewidth=2)
axes[0].plot(ns, wis_bias, 'r-s', label='WIS (有偏)', linewidth=2)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('样本数 N'); axes[0].set_ylabel('偏差')
axes[0].set_title('OIS vs WIS：偏差比较')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ns, ois_var, 'b-o', label='OIS (高方差)', linewidth=2)
axes[1].plot(ns, wis_var, 'r-s', label='WIS (低方差)', linewidth=2)
axes[1].set_xlabel('样本数 N'); axes[1].set_ylabel('方差')
axes[1].set_title('OIS vs WIS：方差比较')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/figures/16_ois_vs_wis.png', dpi=100)
plt.close()
print(f"真实值 E_p[x] = {true_Ep:.4f}")
print(f"OIS: 最终偏差={ois_bias[-1]:.4f}, 方差={ois_var[-1]:.4f}")
print(f"WIS: 最终偏差={wis_bias[-1]:.4f}, 方差={wis_var[-1]:.4f}")
print("✅ 图已保存到 outputs/figures/16_ois_vs_wis.png")


## 重要性权重截断 (Importance Weight Truncation)

### 动机

当重要性权重 $\rho(x)$ 过大时，单个样本就能主导整个估计。截断是一种简单的方差控制方法：

$$\hat{\rho}(x) = \min\left(\rho(x), c\right)$$

### 截断的重要性采样

$$\hat{J}_{\text{Trunc}} = \frac{1}{N} \sum_{i=1}^{N} \min(\rho_i, c) \, f(x_i)$$

- $c$ 是截断阈值（通常 $c \in [1, 10]$）
- **引入偏差**：截断后的估计不再无偏
- **降低方差**：限制了极端权重的影响

### 偏差-方差权衡

| 阈值 $c$ | 偏差 | 方差 | 说明 |
|:--------:|:----:|:----:|:------|
| $c \to \infty$ | 无 | 高 | 普通 IS |
| $c$ 适中 | 小 | 中 | 好的平衡 |
| $c = 1$ | 高 | 低 | 退化为无校正 |

### 与 PPO 裁剪的联系

PPO 的裁剪 $\text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)$ 本质上就是一种**重要性权重截断**，它的截断阈值是 $1 \pm \varepsilon$ 而不是一个固定的 $c$。

**区别**：PPO 的裁剪同时限制上限和下限，而普通的权重截断只限制上限。这是因为在 PPO 中，$\rho = 1$ 是"无更新"的参考点。


In [ ]:
# ============================================================
# 重要性权重截断演示
# ============================================================
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# 构造一个权重分布差异大的场景
mu_p, sigma_p = 3.0, 0.5
mu_q, sigma_q = 0.0, 2.0

def gaussian_pdf(x, mu, sigma):
    return 1.0 / (sigma * np.sqrt(2 * np.pi)) * np.exp(-0.5 * ((x - mu) / sigma)**2)

N = 5000
x_q = np.random.normal(mu_q, sigma_q, N)
rho = gaussian_pdf(x_q, mu_p, sigma_p) / gaussian_pdf(x_q, mu_q, sigma_q)
f_x = x_q ** 2

true_value = mu_p**2 + sigma_p**2  # E[x^2] for N(mu, sigma^2)

# 不同截断阈值
c_values = [1.0, 2.0, 5.0, 10.0, 50.0, 100.0, np.inf]

estimates = []
variances = []

for c in c_values:
    run_estimates = []
    for _ in range(200):
        idx = np.random.choice(N, N, replace=True)
        rho_trunc = np.minimum(rho[idx], c)
        est = np.mean(rho_trunc * f_x[idx])
        run_estimates.append(est)
    estimates.append(np.mean(run_estimates))
    variances.append(np.var(run_estimates))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = [f'c={c}' if c != np.inf else 'c=∞' for c in c_values]
x_pos = np.arange(len(c_values))

axes[0].bar(x_pos, [e - true_value for e in estimates], alpha=0.7)
axes[0].axhline(0, color='red', linestyle='--', label='真实值')
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(labels)
axes[0].set_ylabel('偏差'); axes[0].set_title('截断阈值对偏差的影响')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(x_pos, variances, alpha=0.7, color='orange')
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(labels)
axes[1].set_ylabel('方差'); axes[1].set_title('截断阈值对方差的影响')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle(f'真实值 = {true_value:.4f}', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/figures/16_is_truncation.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/16_is_truncation.png")

print("=== 截断效果对比 ===")
print(f"{'阈值':<10} {'估计值':<12} {'偏差':<12} {'方差':<12}")
for c, e, v in zip(c_values, estimates, variances):
    label = f'c={c}' if c != np.inf else 'c=∞'
    print(f"{label:<10} {e:<12.4f} {e-true_value:<+12.4f} {v:<12.6f}")


## 与 PPO 的联系：裁剪作为隐式 IS 方差控制

### PPO 的核心问题

在 PPO 中，我们想最大化：

$$J(\theta) = \mathbb{E}_{a \sim \pi_{\theta_{\text{old}}}} \left[\frac{\pi_\theta(a \mid s)}{\pi_{\theta_{\text{old}}}(a \mid s)} \cdot A^{\pi_{\theta_{\text{old}}}}(s, a)\right]$$

这里 $\rho_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}$ 正是**重要性权重**。

### PPO 如何缓解 IS 的高方差问题

| 技术 | 解决的问题 |
|------|-----------|
| **裁剪 (Clipping)** | 限制 $\rho$ 在 $[1-\varepsilon, 1+\varepsilon]$ 内 |
| **重要性采样** | 允许复用旧数据（多 epoch 更新） |
| **Advantage 归一化** | 减少优势值的尺度差异 |
| **KL 惩罚/提前停止** | 确保 $\pi_\theta \approx \pi_{\theta_{\text{old}}}$ |

### 裁剪的效果

对于 $A > 0$（好动作）：
- 如果 $\rho > 1+\varepsilon$：策略更新过猛，被裁剪回 $1+\varepsilon$
- 如果 $\rho < 1-\varepsilon$：策略意外降低了该动作的概率，保留低的 $\rho$

对于 $A < 0$（坏动作）：
- 如果 $\rho < 1-\varepsilon$：策略大幅降低坏动作概率，被裁剪回 $1-\varepsilon$
- 如果 $\rho > 1+\varepsilon$：策略增加了坏动作概率，保留高的 $\rho$

### 直觉

> 裁剪防止了重要性权重变得过大或过小，从而**直接限制了 IS 的方差**。PPO 不需要像 TRPO 那样计算 KL 散度或自然梯度，通过简单的裁剪就实现了类似的稳定效果。

这本质上是**将 IS 的方差控制问题转化为一个简单的数值裁剪问题**。


## 本节总结

### 核心要点

| 概念 | 描述 |
|------|------|
| **重要性采样** | 用 $q$ 的样本估计 $\mathbb{E}_p[f(x)]$，通过权重 $\rho(x) = p(x)/q(x)$ 校正 |
| **轨迹级 IS** | $\rho_{0:T-1} = \prod_{t=0}^{T-1} \frac{\pi(a_t \mid s_t)}{\mu(a_t \mid s_t)}$，约掉环境动态 |
| **Per-decision IS** | 每个奖励只乘以到该时刻的权重，降低方差 |
| **高方差问题** | 权重方差随轨迹长度指数增长，ESS 大幅降低 |
| **OIS vs WIS** | OIS 无偏高方差，WIS 有偏低方差 |
| **PPO 的裁剪** | 通过限制权重范围来控制 IS 方差 |

### 关键洞察

重要性采样是 off-policy 学习的数学桥梁，但它的高方差是所有 off-policy 算法必须面对的核心挑战。PPO 的裁剪技巧本质上就是一种简洁有效的 IS 方差控制方法。


## 练习

1. **高斯 IS 实验**：修改演示代码，让 $p$ 和 $q$ 的差异更大（如 $\mu_p=3, \sigma_p=0.5$ 和 $\mu_q=0, \sigma_q=3$），观察权重分布和 ESS 的变化。

2. **WIS vs OIS**：在第 7 节的代码中添加 WIS 估计，比较 OIS 和 WIS 的偏差和方差（多次运行）。

3. **轨迹长度的影响**：在第 8 节的 2 状态 MDP 中，改变 `max_steps`（从 5 到 50），绘制权重方差随轨迹长度的变化曲线。

4. **ESS 诊断**：实现一个函数，给定权重数组计算 ESS。在训练中监控 ESS，当 ESS/N < 0.1 时触发策略回滚或停止更新。

5. **裁剪与 IS 方差**：实现一个实验，比较普通 IS 和裁剪 IS（将权重限制在 $[1-\varepsilon, 1+\varepsilon]$）的方差，验证 PPO 裁剪的方差降低效果。
